# <b> <font color='#A20025'> Appendix I &middot; Notebook 5 of 5: Frontier Read-Alongs: Diffusion at the Protein Scale</font> </b>

*This is a bonus notebook accompanying Appendix I of [Build AI Drug Discovery Pipelines](https://github.com/nrflynn2/ml-drug-discovery). It is self-contained; read it side by side with the appendix. This notebook covers three frontier tools as read-alongs with executed results where we ran them: RFdiffusion, the two directions of structure-based design, and the Boltz-2 connector.*

## <b> <font color='#A20025'>  Environment setup</font> </b>


This notebook is **mostly prose and read-along code**. RFdiffusion and Boltz-2 are **not installed or
run here**; they need a GPU (and, for Boltz-2, 24-48 GB of VRAM). Two libraries do execute: **RDKit**,
to render the FlowMol3 molecule that seeds the &sect;I.7 connector, and **pandas**, to read the cached
results tables for the controlled Boltz-2 comparison and the re-docking cross-check. Both are CPU-only
and fast.

If you are on Colab, this one install is all you need:

In [ ]:
# The runtime dependencies in NB5; everything else is read-along.
# Colab: uncomment the next line.
# !pip install rdkit==2024.9.6 pandas

import rdkit
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, QED, rdMolDescriptors, AllChem
from rdkit.Chem.Draw import rdMolDraw2D
import pandas as pd
import datetime

print("RDKit       :", rdkit.__version__)
print("pandas      :", pd.__version__)
print("Notebook run:", datetime.date.today().isoformat())
print("Frontier tools below are READ-ALONG; not executed in this CPU environment.")

In [ ]:
# --- Path setup: resolve data/figures relative to this notebook, so paths work on Colab and locally ---
from pathlib import Path
import os

# On Colab, clone the repo and move into this notebook's folder on first run.
if not Path("data").exists() and not Path("../data").exists():
    if os.path.exists("/content"):  # Colab
        os.system("git clone --depth 1 https://github.com/nrflynn2/ml-drug-discovery.git")
        os.chdir("ml-drug-discovery/Appendix_I_FLYNN_ML4DD/notebooks")

HERE = Path.cwd()
APPENDIX = HERE.parent if HERE.name == "notebooks" else HERE
DATA = APPENDIX / "data"
FIGURES = APPENDIX / "figures"
ARTIFACTS = APPENDIX / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("data dir:", DATA)

<div style="background:#A20025;padding:11px 16px;border-radius:6px;margin-top:6px;">
<span style="color:#ffffff;font-size:20px;font-weight:700;">&nbsp;&nbsp;Diffusion beyond small molecules: RFdiffusion</span><br><span style="color:#f3d0d8;font-size:13px;font-family:monospace;">§I.2 · concept sidebar</span>
</div>

In NB1 you corrupted a 2-D point cloud with Gaussian noise until it was structureless, then
trained a network to walk it back, step by step, to something that looked like your data. Hold
onto that picture; because **nothing about it was specific to two dimensions, or to points.**

Swap the point cloud for the **3-D coordinates of a protein backbone**; the (x, y, z) of every
Cα, the orientation of every residue frame; and the *exact same recipe* applies. You start from
a real backbone, add noise to the coordinates and orientations until the chain is a random tangle,
and train a network to reverse it. Sample pure noise, run the reverse process, and a **brand-new,
physically plausible protein backbone** condenses out of the fog. That is **RFdiffusion**.

The leap that made it work: the denoiser isn't a generic U-Net. RFdiffusion fine-tunes
**RoseTTAFold**; a network that already *understands* protein structure from the folding problem  - 
into the role of the denoiser. You are not teaching a blank slate what a protein looks like; you
are teaching a structure-fluent model to run its knowledge *backwards*.

<div style="background:#eef4fb;border-left:5px solid #2c6fbf;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#2c6fbf;"> The idea that transfers</b><br>

Small-molecule generators (NB3's FlowMol3) and protein-backbone generators (RFdiffusion) are the
<b>same algorithm at two scales</b>:
<br><br>
&nbsp;&nbsp;• <b>corrupt</b> a real structure with noise → &nbsp;<b>learn to reverse</b> it → &nbsp;<b>sample</b> noise and denoise into something new<br><br>
FlowMol3 denoises an <i>atom cloud</i> (~20-40 atoms, coordinates + element + charge + bond types,
all at once). RFdiffusion denoises a <i>residue chain</i> (hundreds of Cα positions + orientation
frames). The token changes; the denoising loop does not.

</div>

### <b> <font color='#A20025'> What RFdiffusion is actually *for*</font> </b>


De-novo backbone generation is not a party trick; it is the entry point to real design tasks. By
**conditioning** the reverse process (holding part of the structure fixed, or specifying "hotspot"
residues on a target), you drive it toward useful objects:

- **Binder design**; grow a new protein that grips a specified patch on a target of interest.
- **Motif scaffolding**; hold a functional motif (an active site, an epitope) fixed and build a stable protein *around* it.
- **Symmetric assemblies**; generate oligomers with a requested point-group symmetry.

The catch; and the reason this is a sidebar, not a runnable section; is **scale**. RFdiffusion
runs on a GPU, and a real campaign generates *thousands* of backbones to find a handful worth
testing. That is a cluster job, not a Colab cell.

### <b> <font color='#A20025'> Read-along: the canonical RFdiffusion binder-design command</font> </b>


Below is the real thing; the `run_inference.py` call from the RFdiffusion repo's own
`examples/design_ppi.sh`, which designs binders to the insulin receptor. RFdiffusion uses a
**Hydra**-style CLI, so every design choice is a `key=value` override. Read it as a spec of *what
you would ask for*, not as a cell to run.

In [ ]:
# -----------------------------------------------------------------------------
# READ-ALONG; this exact binder-design command needs a target PDB;
# the de-novo form of it (shown below) we DID run on a GPU; see the
# self-consistency loop cells that follow.
# Tool     : RFdiffusion (original)
# Paper    : Watson et al., "De novo design of protein structure and function
#            with RFdiffusion", Nature 620 (2023). doi:10.1038/s41586-023-06415-8
# Code      : github.com/RosettaCommons/RFdiffusion
# License   : BSD (open source); free for non-profit AND for-profit use
# Hardware  : NVIDIA GPU required (backbone generation is a GPU job)
# Retrieved : 2026-07; VERIFY against the repo README before running
# -----------------------------------------------------------------------------

# --- 1. Install (one-time, on a GPU box) --------------------------------------
#   git clone https://github.com/RosettaCommons/RFdiffusion.git
#   cd RFdiffusion
#   conda env create -f env/SE3nv.yml        # SE(3)-equivariant stack
#   conda activate SE3nv
#   # then download the pretrained model weights per the README's script

# --- 2. Design binders to a target (from examples/design_ppi.sh) --------------
#   ./scripts/run_inference.py \
#       inference.output_prefix=example_outputs/design_ppi \
#       inference.input_pdb=input_pdbs/insulin_target.pdb \
#       'contigmap.contigs=[A1-150/0 70-100]' \
#       'ppi.hotspot_res=[A59,A83,A91]' \
#       inference.num_designs=10 \
#       denoiser.noise_scale_ca=0 denoiser.noise_scale_frame=0

# How to READ that command:
#   contigmap.contigs=[A1-150/0 70-100]
#       "Keep residues 1-150 of target chain A, then a chain break (/0),
#        then diffuse a NEW binder of length 70-100 (sampled each run)."
#   ppi.hotspot_res=[A59,A83,A91]
#       "Aim the binder at these three residues on the target surface."
#   inference.num_designs=10
#       "Emit 10 backbones."  (a real campaign asks for thousands)
#   denoiser.noise_scale_ca=0 / noise_scale_frame=0
#       "Drop the extra inference noise -> cleaner, more designable backbones."
print("Read-along only; see github.com/RosettaCommons/RFdiffusion")

Notice what RFdiffusion does **not** give you. Its output is a **backbone**; a trace of Cα
positions and orientations with the sequence left as poly-glycine placeholders. It designs the
*shape*, not the *sequence that folds into it*. Turning that shape into a real, orderable protein
takes more tools; which is exactly the point of §I.6.

### <b> <font color='#A20025'> We ran it: the RFdiffusion → ProteinMPNN → ESMFold self-consistency loop</font> </b>


The read-along above shows the *binder* form of the command. To make the whole idea concrete; and
to answer the question cell 10 raised, *"is this backbone a shape a real sequence could actually
fold into?"*; we ran the full de-novo design loop on an A100. It is the standard way the field
validates a generated backbone, and it has three steps:

1. **RFdiffusion** generates backbones from noise (the *shape*).
2. **ProteinMPNN** designs sequences for each backbone (inverse folding: *shape → sequence*).
3. **ESMFold** folds each designed sequence back, *independently*; and we measure how close the
   refold lands to the original backbone (**self-consistency RMSD**, scRMSD).

If an independent predictor rebuilds the same shape from sequence alone, the backbone is
**designable**. The accepted bar is scRMSD < 2 Å at pLDDT > 70. All the GPU work below ran on
Modal; the cells here **load the saved results** so the notebook renders on CPU in seconds. Set
`USE_CACHED = False` and provide a GPU to re-run from scratch.

In [ ]:
# -- Step 1: RFdiffusion - generate backbones from noise ----------------------
# What we actually ran on the A100 (5 designs, 80 residues, T=50 denoising):
#
#   python scripts/run_inference.py \
#       inference.output_prefix=/work/out/mono \
#       inference.model_directory_path=/weights \
#       inference.input_pdb=input_stub.pdb \
#       inference.num_designs=5 \
#       'contigmap.contigs=[80-80]' \
#       diffuser.T=50
#
# Runtime: ~249 s on one A100 (~$0.14). All 5 came back with zero chain breaks.
# We kept the 3 most compact, distinct folds for design:
#   rfd_backbone_0 : three-helix bundle (Rg 12.7 Å, 96% helix, 3 packed helices)
#   rfd_backbone_1 : single long helix  (Rg 11.7 Å)
#   rfd_backbone_4 : small alpha/beta fold (38% sheet)
#
# The backbone PDBs are saved as chapter artifacts (rfd_backbone_{0,1,4}.pdb).
print("RFdiffusion produced 5 valid backbones; 3 selected for sequence design.")

In [ ]:
# -- Step 2: ProteinMPNN - design sequences for each backbone ------------------
# ProteinMPNN (Dauparas et al., Science 2022) is an *inverse-folding* network:
# it takes a fixed backbone and outputs sequences likely to fold into it.
# What we ran (per backbone), on the same GPU image (ProteinMPNN is baked in):
#
#   python /app/proteinmpnn/protein_mpnn_run.py \
#       --pdb_path rfd_backbone_0.pdb --pdb_path_chains A \
#       --out_folder out/mpnn --num_seq_per_target 8 \
#       --sampling_temp 0.1 --seed 37
#
# 8 sequences x 3 backbones = 24 designs. The best sequence for the
# three-helix bundle (backbone_0), which scored best in step 3, was:
WINNER_SEQ = "HMAAHIAEIEAMSDEELEEKVLKLAKEKLTPEQQKLFIEELEKLKKKDYEKYKELIRKILLEEEEAMEKIKEEIEKKLAA"
print(f"Best designed sequence ({len(WINNER_SEQ)} aa):")
print(WINNER_SEQ)

In [ ]:
# -- Step 3: ESMFold refold + self-consistency RMSD ---------------------------
# Each designed sequence is folded INDEPENDENTLY by ESMFold (Lin et al.,
# Science 2023), which never saw the RFdiffusion backbone. We then align the
# refold to the parent backbone (Kabsch) and report Ca RMSD + pLDDT.
#
# GPU run: ProteinMPNN + 24 ESMFold folds finished in 79 s on an A100 (~$0.05).
import json, os
USE_CACHED = True   # load saved results (CPU-friendly). Set False to re-fold on a GPU.

# Load the saved self-consistency table (shipped as a chapter artifact):
#   selfconsistency_full.json  -> {'all': [...24...], 'best': {...}, 'winner': {...}}
sc_path = str(DATA / "selfconsistency_full.json")
if not os.path.exists(sc_path):
    sc_path = "selfconsistency_full.json"      # fallback: local working copy
results = json.load(open(sc_path))
allr, best, winner = results["all"], results["best"], results["winner"]

labels = {"rfd_backbone_0":"3-helix bundle",
          "rfd_backbone_1":"single helix",
          "rfd_backbone_4":"alpha/beta fold"}
print(f"{'backbone':<16}{'best scRMSD':>12}{'pLDDT':>8}   verdict")
for tag, r in best.items():
    ok = r["scrmsd"] < 2.0 and r["plddt"] > 70
    print(f"{labels[tag]:<16}{r['scrmsd']:>10.2f} A{r['plddt']:>8.0f}   "
          f"{'DESIGNABLE' if ok else 'borderline'}")

n_pass = sum(1 for r in allr if r["scrmsd"] < 2.0 and r["plddt"] > 70)
print(f"\n{n_pass}/{len(allr)} designs pass the designability bar (scRMSD < 2 A, pLDDT > 70).")
print(f"Overall winner: {labels[winner['backbone']]} -> "
      f"scRMSD {winner['scrmsd']} A, pLDDT {winner['plddt']}, TM {winner['tm']}")

The result is emphatic (Figure I.12): **23 of the 24 designs are self-consistent**, and the three-helix bundle's best sequence folds back to within **0.43 Å** Cα RMSD at pLDDT 90. The ESMFold prediction and the RFdiffusion backbone are, for practical purposes, the same structure. That closed loop (generate, design, fold back, and check) is exactly why the field trusts these backbones enough to order the genes. We also scaled the loop across backbone lengths 60 to 150; designability stayed high throughout (58 to 88 percent of designs passing), shown in the right panel of Figure I.12.

![Figure I.12]({{artifact:e7e3627e-9793-4122-83c1-3ada0a7fb2cd}})

**Figure I.12.** RFdiffusion backbones are designable, validated by the ProteinMPNN to ESMFold self-consistency test. Panel (a) overlays a generated backbone with its refold (Cα RMSD 0.43 Å); panel (b) shows the per-residue deviation with a mean of 0.38 Å; panel (c) plots all 24 designs of the 80-residue run in the designable region; and panel (d) shows designability climbing from 58 to 88 percent as backbone length grows from 60 to 150 residues.

<div style="background:#f2f7ee;border-left:5px solid #4a7c2f;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#4a7c2f;"> Forward pointer; the family churns fast (v1 → AA → v2 → v3 in ~2.5 years)</b><br>

The original RFdiffusion (2023) designs <b>protein backbones</b>. The lineage moved quickly:
<br><br>
&nbsp;&nbsp;<b>RFdiffusion</b> (Nature 2023); protein backbones.<br>
&nbsp;&nbsp;<b>RFdiffusionAA</b> (all-atom); reasons over small-molecule atoms too (the §I.6 mirror image).<br>
&nbsp;&nbsp;<b>RFdiffusion2 / RFdiffusion3</b>; a <i>unified all-atom</i> generator over proteins,
nucleic acids, small molecules, and mixed assemblies; close in spirit to FlowMol3's
"all modalities at once," but at the biomolecular-complex scale.
<br><br>
<b>Watch the two dates for RFdiffusion3; they are not the same event:</b><br>
&nbsp;&nbsp;• <b>Preprint posted 2025-09-18</b> (v1); v2 on 2025-11-19  - 
Butcher, Krishna, Mitra, ... Baker, <i>"De novo Design of All-atom Biomolecular Interactions
with RFdiffusion3,"</i> bioRxiv, doi:10.1101/2025.09.18.676967.<br>
&nbsp;&nbsp;• <b>Open-source code (MIT) released ~2025-12</b> (Rosetta Commons Foundry).
<br><br>
If you remembered RFdiffusion3 as a "December 2025" thing; that is the <b>code release</b>, three
months <i>after</i> the science first appeared. Getting that distinction right is the whole reason
this notebook date-stamps everything.

</div>

<div style="background:#A20025;padding:11px 16px;border-radius:6px;margin-top:6px;">
<span style="color:#ffffff;font-size:20px;font-weight:700;">&nbsp;&nbsp;Two directions of structure-based design</span><br><span style="color:#f3d0d8;font-size:13px;font-family:monospace;">§I.6 · note</span>
</div>

Structure-based drug design has a shape and a pocket, and a diffusion model can hold **either one
fixed and generate the other**. That symmetry is worth pausing on, because it is the clearest
demonstration that these models are not one-trick generators; they are *conditional* denoisers,
and you choose what to condition on.

- **DiffSBDD** (your §I.5 capstone); **fix the pocket, generate the ligand.** You hand it a
  protein binding site and it denoises a small molecule *into* that pocket. This is the classic
  hit-generation direction: "given this target, invent me a binder."
- **RFdiffusionAA**; **fix the ligand, generate the pocket.** You hand it a small molecule and it
  builds a *protein* whose binding site is shaped to hold it. This is the direction you take when
  the small molecule is the fixed point; designing a binder, a sensor, or an enzyme around a
  chosen substrate.

Same denoising machinery, mirror-image conditioning. One invents the key for a given lock; the
other builds a lock for a given key.

![Two directions of structure-based design]({{artifact:22a6ea9c-5345-4218-8074-c6c0f1022775}})

**Figure.** Two directions of structure-based design. Left (DiffSBDD): the pocket is the fixed input and the ligand is denoised into it. Right (RFdiffusionAA): the ligand is fixed and a protein pocket is generated around it. The denoiser is the same; only the conditioning flips.

| | **DiffSBDD** | **RFdiffusionAA** |
|---|---|---|
| **You provide (fixed)** | a protein **pocket** | a small-molecule **ligand** |
| **Model generates** | a **ligand** in the pocket | a **protein pocket** around the ligand |
| **Design question** | "invent a binder for this target" | "build a protein to hold this molecule" |
| **Modality generated** | small-molecule atoms + bonds | protein backbone + side-chain atoms |
| **Primary reference** | Schneuing et al., *Nat. Comput. Sci.* 2024 | all-atom RFdiffusion (Baker lab) |
| **License / data note** | MIT code; CrossDocked2020 training data is **GPL-2.0** | see repo; weights terms vary by release |

### <b> <font color='#A20025'> One model designs; a *pipeline* validates</font> </b>


Here is the part that a single impressive demo hides: **generating a backbone is the first step,
not the finished product.** RFdiffusion gives you a *shape* with no sequence. Before that shape is
anything you would order and test, it goes through a chain of specialist tools; each answering a
question the generator cannot:

1. **RFdiffusion** → *generate the backbone.* What 3-D shape grips the target?
2. **ProteinMPNN / LigandMPNN** (inverse folding) → *find a sequence.* Which amino-acid sequence
   actually **folds into** that backbone? (LigandMPNN is the variant that accounts for a bound
   ligand or other non-protein atoms.)
3. **AlphaFold2 / RoseTTAFold** (refold & validate) → *does it hold up?* Fold the designed sequence
   from scratch and check it lands back on the shape you asked for; high pLDDT and low
   backbone RMSD to the design is the standard *in-silico* pass/fail.
4. **PyRosetta** (physics-based scoring) → *is it energetically sane?* Score the interface, relax
   the structure, rank candidates by binding energy before anything reaches a wet lab.

Only the designs that survive **all four** stages are worth synthesizing. This is the single most
important mindset shift in the chapter's frontier material: **de-novo design is a pipeline, not a
model.** The generator is one component; the validation loop around it is what makes the output
trustworthy.

<div style="background:#eef4fb;border-left:5px solid #2c6fbf;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#2c6fbf;"> Where to go hands-on; DL4Proteins (Graylab)</b><br>

Standing up a GPU box to run this full stack is beyond this appendix's CPU scope, but you don't have
to assemble it from scratch. The <b>DL4Proteins</b> notebook series from the Graylab
(<span style="font-family:monospace;">github.com/Graylab/DL4Proteins-notebooks</span>) walks the
whole loop end-to-end and is the recommended companion for this section:
<br><br>
&nbsp;&nbsp;• <b>WS8</b>; RFdiffusion for backbone generation.<br>
&nbsp;&nbsp;• <b>WS9</b>; RFdiffusion → ProteinMPNN → AlphaFold2, the design-and-validate loop.<br>
&nbsp;&nbsp;• <b>WS10</b>; binder design pulling the tools together.
<br><br>
Read them alongside this note: they are the runnable, GPU-backed version of the pipeline sketched
above. <i>(Retrieved 2026-07; check the repo for the current notebook numbering.)</i>

</div>

<div style="background:#A20025;padding:11px 16px;border-radius:6px;margin-top:6px;">
<span style="color:#ffffff;font-size:20px;font-weight:700;">&nbsp;&nbsp;From generation to triage: Boltz-2</span><br><span style="color:#f3d0d8;font-size:13px;font-family:monospace;">§I.7 · pipeline connector</span>
</div>

You can now *generate* molecules; FlowMol3 hands you valid 3-D ligands, DiffSBDD hands you ligands
shaped for a specific pocket. But generation raises an immediate, unforgiving question:
**which of these actually bind?** A generator will happily emit a hundred chemically-valid
molecules; most of them will not touch your target. You need a triage step.

That is the connector. The pipeline is:

> **generate** ligands (FlowMol3 / DiffSBDD) → **co-fold** each one with the target protein and
> **predict its binding affinity** (Boltz-2) → **rank and triage** → carry the top few forward.

**Boltz-2** is the co-fold-and-score step. Given a protein sequence and a ligand, it predicts the
**3-D structure of the complex** *and*; through a dedicated affinity head; **how strongly the
ligand binds**. It is an open (MIT) re-implementation in the AlphaFold3 lineage, extended with the
affinity prediction that makes it useful for hit triage.

Let's make the input concrete. Below is a **real molecule from this appendix's FlowMol3 run** (NB3)  - 
the exact SMILES that run emitted. This render **is executed** (RDKit runs on CPU); it is the
tangible object that flows into the read-along Boltz-2 block that follows.

In [ ]:
# RUNNABLE; this is the concrete ligand handed off from the FlowMol3 run (NB3).
flowmol3_smiles = "Cc1ccc(NC(=O)c2ccc(OC(F)(F)F)cc2)cc1"   # a trifluoromethoxy benzamide
mol = Chem.MolFromSmiles(flowmol3_smiles)

profile = {
    "Canonical SMILES": Chem.MolToSmiles(mol),
    "Formula"         : rdMolDescriptors.CalcMolFormula(mol),
    "Mol. weight"     : f"{Descriptors.MolWt(mol):.2f} g/mol",
    "QED"             : f"{QED.qed(mol):.3f}",
    "cLogP"           : f"{Descriptors.MolLogP(mol):.2f}",
    "H-bond donors"   : rdMolDescriptors.CalcNumHBD(mol),
    "H-bond acceptors": rdMolDescriptors.CalcNumHBA(mol),
    "Heavy atoms"     : mol.GetNumHeavyAtoms(),
}
for k, v in profile.items():
    print(f"{k:18s}: {v}")

# draw it
img = Draw.MolToImage(mol, size=(520, 360))
img   # displays inline in the notebook

This molecule (C₁₅H₁₂F₃NO₂, MW ≈ 295, QED ≈ 0.92, well within Lipinski limits) is a plausible-looking
hit. But "plausible-looking" is not "binds." To triage it, you co-fold it against your target and
read off an affinity; and that is Boltz-2's job.

![Figure I.13]({{artifact:3ecbd280-19a7-4cad-8a46-3731f62c189e}})

**Figure I.13.** The FlowMol3-generated molecule that we hand to Boltz-2 as the concrete ligand input. The 2D depiction shows a trifluoromethoxy benzamide (C15H12F3NO2, molecular weight about 295, QED about 0.92) alongside its computed molecular profile. This is a real de novo output from the FlowMol3 run in Notebook 3, carried forward here as the ligand in the co-folding and affinity prediction.

### <b> <font color='#A20025'> Read-along: co-folding + affinity with Boltz-2</font> </b>


Boltz-2 takes a small **YAML** describing the complex you want folded, plus an optional
`properties: affinity` block that switches on the affinity head. Below is that YAML, wired up with
the FlowMol3 molecule above as the ligand, followed by the CLI that runs it. We **executed this on a Modal A100** for the chapter (161 s), using a precomputed AlphaFold DB
alignment for the MSA; every field and number below is real and version-pinned.

In [ ]:
# -----------------------------------------------------------------------------
# EXECUTED ON GPU (Modal A100); real §10.7 numbers, MSA-backed
# Tool     : Boltz-2   (structure prediction / co-folding + binding affinity)
# Paper    : Passaro et al., "Boltz-2: Towards Accurate and Efficient Binding
#            Affinity Prediction", bioRxiv (posted 2025-06-18).
#            doi:10.1101/2025.06.14.659707
# Code      : github.com/jwohlwend/boltz
# Version   : boltz 2.2.1 (PyPI)   License: MIT
# Hardware  : NVIDIA GPU, ~24-48 GB VRAM (affinity head is memory-heavy)
# Retrieved : 2026-07; VERIFY the YAML schema against docs/prediction.md before running
# -----------------------------------------------------------------------------

# --- 1. Install (on a GPU box) ------------------------------------------------
#   pip install boltz==2.2.1

# --- 2. Get a precomputed MSA for a catalogued target -------------------------
#    Boltz sharpens the fold with a multiple-sequence alignment. Rather than call
#    the (often-overloaded) ColabFold server, download the alignment the AlphaFold
#    Database already built for this UniProt entry; instant, deep, no server:
#
#      https://alphafold.ebi.ac.uk/files/msa/AF-<UNIPROT>-F1-msa_v6.a3m
#
#    For A2A (UniProt P29274) this is a 17,819-sequence .a3m. Feed it via the
#    YAML `msa:` field (no --use_msa_server needed).
#
#   import urllib.request
#   urllib.request.urlretrieve(
#       "https://alphafold.ebi.ac.uk/files/msa/AF-P29274-F1-msa_v6.a3m",
#       "a2a.a3m")

# --- 3. Describe the complex: target protein + your FlowMol3 ligand -----------
#    Save as  complex.yaml  (schema per boltz docs/prediction.md):
boltz_yaml = """
version: 1
sequences:
  - protein:
      id: A
      sequence: MPIMGSSVYITVELAIAVLAILGNVLVCWAVWLNSNLQNVTNYFVVSLAAAD...   # A2A, P29274 (412 aa)
      msa: a2a.a3m           # precomputed AlphaFold DB alignment (17,819 seqs)
                             # use  msa: empty  for single-sequence (faster, lower accuracy)
  - ligand:
      id: B
      smiles: 'Cc1ccc(NC(=O)c2ccc(OC(F)(F)F)cc2)cc1'   # <- FlowMol3 output (NB3)

properties:
  - affinity:
      binder: B            # ask for the affinity of ligand chain B to protein A
"""
# NB: exactly ONE ligand chain may be scored for affinity, and it must be small
#     (<=128 heavy+kept-H atoms; the authors recommend <=~56).  Our ligand: 21 heavy atoms.

# --- 4. Run co-folding + affinity ---------------------------------------------
#   boltz predict complex.yaml --out_dir out/ --recycling_steps 3 --diffusion_samples 5
#
#   Key flags (from docs/prediction.md):
#     --use_msa_server            auto-generate the protein MSA (else provide your own .a3m via msa:)
#     --recycling_steps           default 3
#     --diffusion_samples         default 5

# --- 5. What we got (Modal A100, 161 s, MSA-backed) ---------------------------
#   Structure : pTM 0.92, ligand-interface ipTM 0.95, complex pLDDT 0.84
#   Affinity  : binder probability 0.09, i.e. BELOW the 0.5 gate -> not called a binder
#               (the regression head read log10(IC50/uM) = 1.17; we do NOT convert that
#                to a potency, because the gate is shut. See the caution callout below.)
#   Read      : high-confidence fold, negative affinity call; a clean triage that flags
#               this drug-like-but-untargeted molecule as a weak A2A starting point.
#   (Single-sequence mode gave binder probability 0.15 with lower fold confidence; the
#    MSA raised structural certainty without moving the molecule past the gate.)
print("Executed on GPU; see boltz_a2a_msa_summary.json for the full result")


### <b> <font color='#A20025'> Reading the output</font> </b>

Boltz-2 writes an `affinity_[name].json` alongside the predicted complex (`*_model_0.cif`), and the two
numbers you triage on are below; these are the **real values from our MSA-backed Modal run**, not
placeholders. Read them in the right order. The **binary head** is the gate: at a binder probability of
**0.09** it does not call this molecule a binder. The **structural** head is confident anyway (pTM 0.92,
ligand-ipTM 0.95). So Boltz-2 is telling us it knows where this ligand sits and doubts that it binds, and
the regression head's log<sub>10</sub>(IC<sub>50</sub>) value stays uninterpreted because the gate is
shut. Held together, that split verdict is the whole point of a triage step.

In [ ]:
# REAL OUTPUT from the MSA-backed Boltz-2 run (Modal A100, 161 s).
# Boltz-2 writes affinity_[name].json next to the predicted complex *_model_0.cif.
msa_backed_affinity = {
    'affinity_pred_value':         1.166,   # log10(IC50 in uM) from the REGRESSION head
    'affinity_probability_binary': 0.093,   # P(binder), 0..1, from the BINARY head
    'ptm':                         0.923,   # structural confidence (whole complex)
    'ligand_iptm':                 0.949,   # interface confidence (pocket-ligand)
}

# Read the BINARY head first: it is the gate that decides whether the regression
# head means anything at all. See the caution callout below for why this ordering
# is not optional.
BINDER_GATE = 0.5
prob = msa_backed_affinity['affinity_probability_binary']
is_binder = prob > BINDER_GATE

print(f"P(binder)        : {prob:.2f}  ->  {'binder' if is_binder else 'NOT called a binder'}")
print(f"pTM / ligand-ipTM: {msa_backed_affinity['ptm']:.2f} / {msa_backed_affinity['ligand_iptm']:.2f}")

if is_binder:
    print(f"Predicted IC50   : {10 ** msa_backed_affinity['affinity_pred_value']:.1f} uM")
else:
    print(f"log10(IC50/uM)   : {msa_backed_affinity['affinity_pred_value']:.2f} "
          f"(reported as-is; NOT converted to an IC50, because the binary head "
          f"did not call this molecule a binder)")

# How to USE the two heads (per the Boltz docs):
#   affinity_probability_binary -> binder vs decoy triage (hit discovery). THE GATE.
#   affinity_pred_value         -> rank actives against each other (hit-to-lead),
#                                  meaningful only for molecules past the gate.

![Figure I.14]({{artifact:ef6eb59a-496f-483b-a080-358b3848419d}})

**Figure I.14.** Boltz-2 triage of the FlowMol3 molecule against the A2A receptor, MSA-backed. Panel (a)
shows all five sampled complexes scoring high and stable structural confidence, with ligand-interface ipTM
holding near 0.95 across samples, so the predicted pose is not an artifact of one lucky draw. Panel (b)
shows the affinity verdict: a binder probability of 0.09 against the 0.5 gate, which is a clear negative
call. The regression head's value is printed beside it as log<sub>10</sub>(IC<sub>50</sub>/uM) rather than
as a micromolar potency, because that head is only defined for molecules the gate admits and this molecule
is not one of them. The split between the panels is the lesson: a confidently modeled pose does not imply a
binder, which is exactly why Boltz-2 serves as a triage connector rather than a generation anchor.

![Figure I.15]({{artifact:b3199fde-d2ed-44db-870e-01b53a27ba9a}})

**Figure I.15.** The Boltz-2 co-folded complex behind the triage score. The adenosine A2A receptor is drawn as a gray cartoon with a translucent surface over the binding pocket, and the FlowMol3 trifluoromethoxy benzamide sits inside it as sticks colored by element (green carbon, blue nitrogen, red oxygen, pale-cyan fluorine). This is the structure the numbers in Figure I.14 summarize: Boltz-2 is confident about where the ligand rests, even as its affinity head stays skeptical that it binds tightly. The interactive complex is available as `data/boltz_a2a_complex.cif` if you want to rotate it in a 3D viewer.

### <b> <font color='#A20025'> One molecule cannot answer the question</font> </b>


The run above scores **one** ligand and returns a low binder probability. That is a result, but it is
not yet an answer, because two different explanations fit it equally well. Maybe the FlowMol3 molecule
really is a poor A2A binder. Or maybe Boltz-2's affinity head is pessimistic about everything you hand
it, in which case the number tells you about the scorer rather than the molecule. With a single
measurement you cannot tell those apart.

The question the pipeline actually needs answered is comparative: **did pocket conditioning help?**
DiffSBDD saw the A2A pocket while it generated (NB4); FlowMol3 did not (NB3). If conditioning works,
the pocket-conditioned ligands should score above the unconditioned one. Answering that takes three
things the single run lacks:

1. a **positive control**, a molecule known to bind this pocket, so you can see whether the scorer can recognise a binder at all
2. **several** generated ligands rather than one, so a single unlucky sample does not decide the verdict
3. a **matched configuration**, so that every score differs only in the ligand and never in the protein, the alignment, or the sampling budget

So we ran it that way: **seven ligands** against a byte-identical protein block (A2A, UniProt P29274,
412 residues) with the same AlphaFold-DB alignment, and we ran the whole set **twice on two independent
Boltz-2 implementations**. The first is the version-pinned open build (boltz 2.2.1) on a Modal A100. The
second is NVIDIA's hosted Boltz-2 NIM, a managed API serving the same model. Running both is a cheap
insurance policy: if two implementations disagree, any conclusion you draw is really a conclusion about
one of them.

In [ ]:
# Controlled Boltz-2 comparison: 7 ligands, ONE protein block, TWO implementations.
# USE_CACHED = True reads the results table committed with this appendix (instant, no GPU).
# USE_CACHED = False would re-run the sweep, which needs a GPU box for the pinned build
# and an API key for the hosted NIM; the read-along block above has the exact command.
USE_CACHED = True

import io, os
import pandas as pd


def read_provenanced_csv(path):
    """Read a CSV that may carry leading '#' provenance lines.

    Skip those lines by hand. Do NOT pass comment='#' to pandas: a '#' also means a
    triple bond inside a SMILES string, so pandas would treat the rest of such a row
    as a comment and silently truncate it to NaN. One ligand in this table (DiffSBDD
    #16, which contains a nitrile) is affected.
    """
    lines = open(path).read().splitlines()
    i = 0
    while i < len(lines) and lines[i].lstrip().startswith("#"):
        i += 1
    return pd.read_csv(io.StringIO("\n".join(lines[i:])))


if USE_CACHED:
    csv_path = str(DATA / "boltz2_comparison.csv")
    if not os.path.exists(csv_path):
        csv_path = "boltz2_comparison.csv"        # fallback: local working copy
    cmp_df = read_provenanced_csv(csv_path)
    print(f"loaded {len(cmp_df)} scored ligands from {os.path.basename(csv_path)}")
else:
    raise NotImplementedError(
        "Fresh scoring needs a GPU (pinned build) or an NVIDIA API key (hosted NIM). "
        "Keep USE_CACHED = True to read the committed results.")

LABEL = {
    "caffeine_control":  "caffeine (3RFM co-crystal)",
    "diffsbdd_3rfm_0":   "DiffSBDD #0",
    "diffsbdd_3rfm_2":   "DiffSBDD #2 (Figure I.10)",
    "diffsbdd_3rfm_16":  "DiffSBDD #16",
    "diffsbdd_3rfm_18":  "DiffSBDD #18",
    "flowmol3_ref":      "FlowMol3 ligand (unconditioned)",
    "diffsbdd_3rfm_4":   "DiffSBDD #4",
}

table = (cmp_df.assign(ligand=cmp_df.mol_id.map(LABEL))
              [["ligand", "qed",
                "modal_affinity_probability_binary",
                "nim_affinity_probability_binary",
                "modal_rank", "nim_rank"]]
              .rename(columns={"qed": "QED",
                               "modal_affinity_probability_binary": "pinned",
                               "nim_affinity_probability_binary": "hosted NIM",
                               "modal_rank": "rank (pinned)",
                               "nim_rank": "rank (NIM)"}))

print()
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

BINDER_GATE = 0.5
for col, name in [("modal_affinity_probability_binary", "pinned build"),
                  ("nim_affinity_probability_binary", "hosted NIM ")]:
    n_pass = int((cmp_df[col] > BINDER_GATE).sum())
    which = ", ".join(cmp_df.loc[cmp_df[col] > BINDER_GATE, "mol_id"])
    print(f"\n{name}: {n_pass} of {len(cmp_df)} ligands clear the {BINDER_GATE} binder gate ({which})")

Read the table top to bottom, because the ordering is the result. **Caffeine, the ligand actually
present in the 3RFM co-crystal, comes first under both implementations** (0.6219 pinned, 0.6172 hosted).
It is also the only one of the seven that clears the 0.5 binder gate, and it clears it twice. That is
the control doing its job: the affinity head is oriented correctly. It is not simply pessimistic about
every molecule you show it, because when a real binder arrives it says so.

Next come the pocket-conditioned ligands. The best DiffSBDD molecule reaches **0.4440**, and the
unconditioned FlowMol3 ligand sits at **0.0906**. Conditioning on the pocket therefore bought a
**4.9-fold higher binder probability**, and the effect is not marginal; four of the five DiffSBDD
ligands outscore the unconditioned one. So pocket conditioning helped, on this metric, in the direction
the method promises.

Now the part that matters more. **0.4440 is still below 0.5.** Every generated ligand, conditioned or
not, sits on the wrong side of the gate under both implementations. The honest reading is that
conditioning **moved the ranking without producing a molecule Boltz-2 calls a binder**. Those are
genuinely different claims, and the second one is what a practitioner needs to hear before ordering
anything. Note also that QED and binder probability are close to unrelated here: the FlowMol3 ligand has
the highest QED in the table (0.9236) and the second-lowest binder probability. Drug-likeness and
predicted binding are separate axes, and a generator can be good at one while indifferent to the other.

In [ ]:
# Do the two implementations agree? If they do not, every conclusion above is a
# conclusion about ONE implementation rather than about Boltz-2.
pin = cmp_df["modal_affinity_probability_binary"]
nim = cmp_df["nim_affinity_probability_binary"]

print(f"Spearman rho (rank agreement) : {pin.corr(nim, method='spearman'):.4f}")
print(f"Pearson r    (value agreement): {pin.corr(nim):.4f}")
print(f"mean |probability difference| : {(nim - pin).abs().mean():.4f}")
print(f"identical ranks               : "
      f"{int((cmp_df.modal_rank == cmp_df.nim_rank).sum())} of {len(cmp_df)}")

swap = cmp_df.loc[cmp_df.modal_rank != cmp_df.nim_rank, "mol_id"].tolist()
gap = float(pin[cmp_df.mol_id == swap[0]].iloc[0] - pin[cmp_df.mol_id == swap[1]].iloc[0])
print(f"\nthe one rank swap             : {swap[0]} vs {swap[1]}, "
      f"separated by {abs(gap):.4f} in the pinned build")

# Wall clock, as measured, for the same 7 ligands.
tot_pin, tot_nim = cmp_df.modal_wall_s.sum(), cmp_df.nim_wall_s.sum()
print(f"\npinned build : {tot_pin:.0f} s over 7 ligands "
      f"({cmp_df.modal_wall_s.min():.0f}-{cmp_df.modal_wall_s.max():.0f} s each)")
print(f"hosted NIM   : {tot_nim:.0f} s over 7 ligands "
      f"({cmp_df.nim_wall_s.min():.0f}-{cmp_df.nim_wall_s.max():.0f} s each)")
print(f"the hosted endpoint was {tot_pin / tot_nim:.1f}x faster on total ligand wall clock")

The two implementations agree closely. Rank correlation is **0.9643**, value correlation **0.9817**, and
the mean absolute difference in binder probability is **0.0290**, so five of the seven ligands hold
exactly the same rank. The single swap is between DiffSBDD #0 and #2, which the pinned build separates by
**0.0049**; that is well inside the spread you would expect from a five-sample diffusion ensemble, so
calling either one "second best" is over-reading the number. The conclusion that survives is the one we
care about: **the ordering is a property of the model, not of the machine you ran it on**, and the gate
verdict is identical under both.

One configuration difference is worth stating plainly rather than burying. The pinned run consumed the
full **17,819-sequence** AlphaFold-DB alignment; the hosted NIM received only the **top 1,024 records**,
because the deeper payload risked the endpoint's size limit. Alignment depth is exactly the kind of input
that can move a structure prediction, so this is a real difference and not a cosmetic one. We record it
because the agreement above was measured *despite* it, which arguably makes the agreement more
impressive rather than less.

The speed gap is the practical trade. The hosted endpoint returned the same seven ligands in **237 s**
against the pinned build's **1065 s**, about **4.5 times faster** on total wall clock, and it needed no
GPU of your own. Against that: you send your molecules to someone else's machine, you cannot pin the
served build (the API does not expose its version), and the licence terms change underneath you, which is
the subject of the callout below.

![Figure I.16]({{artifact:1d6a0aa8-3a53-4747-a7c7-e8b4bcd1844e}})

**Figure I.16.** The controlled Boltz-2 comparison, seven ligands scored twice under a matched
configuration. Panel (a) plots binder probability for each ligand under the version-pinned open build
(crimson) and NVIDIA's hosted Boltz-2 NIM (slate), with the 0.5 binder gate drawn as a dashed line; only
caffeine, the ligand actually present in the 3RFM co-crystal, rises above it, and it does so under both
implementations. The pocket-conditioned DiffSBDD ligands cluster below the gate but clearly above the
unconditioned FlowMol3 molecule, which is the measurable benefit of conditioning; the best conditioned
ligand reaches 0.4440 against 0.0906 unconditioned. Panel (b) plots the hosted probabilities against the
pinned ones, where the points track the diagonal at a Spearman rho of 0.964, so the ranking does not
depend on which implementation produced it. Read together, the panels say that conditioning improved the
ordering while leaving every generated molecule on the wrong side of the binder gate.

![Figure I.17]({{artifact:43ca5820-c150-42c5-923e-67d13a307b91}})

**Figure I.17.** The top-scoring generated ligand where Boltz-2 places it, rendered from the predicted
complex. Panel (a) shows DiffSBDD #0 (crimson sticks) seated in the A2A pocket, with the protein as a
pale cartoon and every side chain within 4.5 angstroms drawn in slate, so you can see that the pocket
closes around the ligand rather than leaving it loose in a cavity. Panel (b) overlays the same ligand on
caffeine from the 3RFM co-crystal (teal) after aligning the two proteins, and the two occupy the same
subpocket with a partial ring overlap. This is the structural reason the affinity head is willing to
score the generated ligand at all: it is in the right place, engaging the right residues. It is also the
limit of what the picture can tell you, because occupying the right pocket is necessary for binding and
nowhere near sufficient, which is precisely the gap the 0.4440 probability is reporting.

<div style="background:#fdf3e7;border-left:5px solid #c9791b;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#c9791b;"> Caution; this is a RANKING result, not a potency prediction</b><br>

Boltz-2's affinity output has <b>two heads, and they are not interchangeable</b>.
<br><br>
&nbsp;&nbsp;<b>1. <code>affinity_probability_binary</code></b>; the binder-vs-decoy classifier. This is
the <b>gate</b>. A value above roughly 0.5 means the model is willing to call the molecule a binder.<br>
&nbsp;&nbsp;<b>2. <code>affinity_pred_value</code></b>; a regression head whose units are
<b>log<sub>10</sub>(IC<sub>50</sub> in micromolar)</b>. It is trained to rank <i>actives against other
actives</i>, and it is only meaningful for molecules that <b>already passed the gate</b>.
<br><br>
In the table above, <b>not one generated ligand passes the gate</b> under either implementation. So
taking any generated ligand's <code>affinity_pred_value</code>, raising 10 to that power, and printing
"predicted IC<sub>50</sub> = <i>x</i> micromolar" would be <b>reporting a number outside the range where
it is defined</b>. The head will always return a value; that is not the same as the value meaning
something. It is a tempting mistake precisely because the output looks like a potency.
<br><br>
<b>What you may say about these results:</b> caffeine ranks first, pocket-conditioned ligands rank above
the unconditioned one, and the ordering reproduces across two implementations. <b>What you may not
say:</b> that any generated ligand has a predicted IC<sub>50</sub>, or that DiffSBDD #0 is "a
0.4-micromolar hit." The gate is shut; downstream potency language stays off the table until something
passes it.

</div>

### <b> <font color='#A20025'> Why Boltz-2 is a *connector*, not an *anchor*</font> </b>


It would be easy to mistake Boltz-2 for another generator to build the chapter on. It is not; and
the distinction is exactly the reasoning behind this appendix's tool choices.

- **Boltz-2 does not generate molecules.** It does **structure prediction / co-folding** (given the
  molecules, predict the complex) plus **affinity prediction** (score the binding). It is the wrong
  task for a *de-novo generation* anchor; it never invents a ligand. It is the *right* task for a
  **connector**: it consumes what your generator produced and tells you which outputs to keep.
- **The generators are the spine; Boltz-2 links them to a decision.** FlowMol3 and DiffSBDD sit at
  the reproducible core of the chapter because they generate, they are small, and they are stably
  licensed. Boltz-2 attaches to the *end* of that spine as the triage stage; a connector between
  "here are candidate molecules" and "here are the two worth testing."

<div style="background:#fdf3e7;border-left:5px solid #c9791b;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#c9791b;"> The churn Boltz-2 illustrates; and why it stays a read-along</b><br>

Boltz-2 is the textbook case for this appendix's read-along discipline. The original
<b>Boltz-2 is MIT-licensed and open</b> (github.com/jwohlwend/boltz); genuinely usable. But it has
already spawned a <b>closed-source, API-only "Boltz-2.1"</b>: the frontier version moves behind a
paywall/API while the open checkpoint you can actually run risks lagging behind.
<br><br>
That is <i>precisely</i> the failure mode a book must not anchor on; a tool whose "current" version
is a moving, partly-closed target. So Boltz-2 earns its place as a <b>version-pinned, dated
connector</b> (boltz 2.2.1, MIT, retrieved 2026-07), not as a load-bearing anchor. If you run it,
run the open MIT release and pin the version; treat anything newer as something to re-verify, not to
assume.
<br><br>
<b>The lesson generalizes:</b> the more popular and fast-moving a tool, the more it belongs in a
read-along; pinned, dated, and kept at arm's length from your reproducible core.

</div>

### <b> <font color='#A20025'> The two heads answer different questions</font> </b>


The controlled run gives us something the single-molecule run could not: evidence about how to *read*
Boltz-2's affinity output. It is worth being precise here, because this is the single easiest place in the
whole pipeline to produce a confident, wrong number.

**The binary head is a gate, and it is the one that decides.** `affinity_probability_binary` answers a
classification question: is this molecule a binder or a decoy? Caffeine, a molecule genuinely co-crystallised
with A2A, scores 0.6219 and 0.6172 under two independent implementations. That is the observation that
licenses everything else, because it shows the head is **oriented correctly**. A scorer that returned low
probabilities for all seven molecules would be indistinguishable from a scorer that is simply pessimistic,
and we would have no way to interpret the generated ligands at all. The control is not decoration; without
it the other six numbers are uninterpretable.

**The regression head is a ranker, and it only speaks past the gate.** `affinity_pred_value` answers a
different question: among molecules that *do* bind, which bind more tightly? Its units are
log<sub>10</sub>(IC<sub>50</sub> in micromolar), and it is trained on actives. Ask it about a molecule the
binary head rejected and it still answers, fluently, in units that look like potency. That fluency is the
trap. The number is outside the domain the head was trained to describe, so its apparent precision is
borrowed, not earned.

Put those together and the correct pipeline shape falls out. **Gate first, rank second.** Use the binary
head to decide what deserves attention; use the regression head only to order what survived. In this run
nothing survived, so the regression column is simply not part of the result, and the deliverable is a
**ranking with a negative verdict**: conditioning improved the order, and the best generated molecule still
did not clear the bar.

This is also why the connector framing from the previous section matters more than it may have seemed.
A connector that returns a clean number for every input is dangerous in exact proportion to how convenient
that number is. Boltz-2 is useful here **because** its gate stayed shut; a triage step that never says no
is not triage.

<div style="background:#f2f7ee;border-left:5px solid #4a7c2f;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#4a7c2f;"> The same model, two sets of rights; check before you ship</b><br>

The two Boltz-2 implementations in this section produce nearly identical science and come with
<b>materially different permissions</b>, which is worth knowing before you build on either.
<br><br>
&nbsp;&nbsp;<b>The repository build</b> (github.com/jwohlwend/boltz, boltz 2.2.1) is <b>MIT</b>, for
academic and commercial use, and you can pin its version.<br>
&nbsp;&nbsp;<b>The hosted NIM</b> is governed by <b>NVIDIA's service terms</b>, not by the repo's MIT
licence. The trial terms scope use to internal testing and evaluation absent a paid subscription, and the
model licence restricts using outputs to develop or improve other AI models without written approval. The
served build's version is also not exposed by the API, so you cannot pin it.
<br><br>
The DiffDock endpoint used in the re-docking cross-check carries the same distinction plus a sharper one:
<b>the weights it serves are not the MIT repository's weights</b>. Its model card records NVIDIA training
on PLINDER and SAIR, with the MIT link appearing as architecture provenance.
<br><br>
<b>The general rule:</b> a licence attaches to a <b>distribution channel</b>, not to a model's name. The
same model reached two ways can carry two sets of obligations, and hosted convenience is one of the things
you trade for. Read the terms at the endpoint you actually call. (Table I.1 in the appendix text records
code and weights licences separately for every model used here, including one case, FlowMol3's weights,
where <b>no licence is stated at all</b>.)

</div>

<div style="background:#A20025;padding:11px 16px;border-radius:6px;margin-top:6px;">
<span style="color:#ffffff;font-size:20px;font-weight:700;">&nbsp;&nbsp;A second opinion: re-docking the generated ligands</span><br><span style="color:#f3d0d8;font-size:13px;font-family:monospace;">&sect;I.7 &middot; independent cross-check</span>
</div>

In NB4 you put the twenty DiffSBDD ligands through PoseBusters and **fifteen of twenty passed** the
full protein-aware physical bar. That result is about whether each pose is *physically admissible*: no
clashes with the protein, sane bond geometry, a ligand that stays inside the structure. It is a
necessary check, and it is silent on a different question that follows immediately from it.

PoseBusters asks whether the pose is *possible*. It does not ask whether the pose is *right*. DiffSBDD
proposed both the molecule and its position in one shot, so nothing has yet checked that position against
an independent opinion. The natural test is a **re-dock**: take each generated molecule as a bare
2-D structure, throw away the coordinates DiffSBDD chose, and ask a separate docking program to place it
in the same receptor from scratch. If the two methods land in the same place, the placement has
independent support. If they do not, you have learned that the pose is one method's opinion.

We ran that with **DiffDock-L**, a diffusion-based blind docking model, through the hosted NVIDIA NIM
endpoint: all twenty ligands, ten poses each, into the same 3RFM receptor DiffSBDD was conditioned on.
Blind docking means we did not tell it where the pocket is, which makes the site agreement below a real
finding rather than something we handed it.

In [ ]:
# DiffDock-L re-docking cross-check: 20 DiffSBDD ligands, 10 poses each, same receptor.
# Cached by default; the fresh path needs the hosted DiffDock NIM (API key + network).
USE_CACHED = True

# Centroid distance for the GENERATED poses, measured in NB4 against the same reference
# ligand. Carried here as a constant because it belongs to the NB4 run, not this table.
GENERATED_CENTROID_MEDIAN_A = 0.632

if USE_CACHED:
    redock_path = str(DATA / "diffdock_redock_posebusters.csv")
    if not os.path.exists(redock_path):
        redock_path = "diffdock_redock_posebusters.csv"     # fallback: local working copy
    rd = read_provenanced_csv(redock_path)                  # same '#'-safe reader as above
    print(f"loaded {len(rd)} re-docked ligands from {os.path.basename(redock_path)}")
else:
    raise NotImplementedError(
        "Fresh re-docking needs the hosted DiffDock NIM (API key). "
        "Keep USE_CACHED = True to read the committed results.")

# 1. Does DiffDock reproduce the generator's pose?
print("\n-- pose agreement (symmetry-corrected RMSD, no alignment: same receptor frame) --")
print(f"median RMSD, DiffDock top-ranked pose vs DiffSBDD pose : "
      f"{rd.rmsd_generated_vs_redocked_A.median():.2f} A")
print(f"median RMSD, closest of the 10 DiffDock poses          : "
      f"{rd.min_rmsd_any_pose_A.median():.2f} A")
print(f"ligands within 2 A using the top-ranked pose           : "
      f"{int((rd.rmsd_generated_vs_redocked_A < 2).sum())} of {len(rd)}")
print(f"ligands within 2 A allowing ANY of 10 poses            : "
      f"{int((rd.min_rmsd_any_pose_A < 2).sum())} of {len(rd)}")

# 2. Do they at least agree on WHERE the site is?
print("\n-- site agreement (ligand centroid distance to the 3RFM co-crystal caffeine) --")
print(f"DiffSBDD poses, median      : {GENERATED_CENTROID_MEDIAN_A:.2f} A   (measured in NB4)")
print(f"DiffDock re-docks, median   : {rd.redock_centroid_dist_to_caffeine_A.median():.2f} A")
print(f"re-docks within 5 A of it   : "
      f"{int((rd.redock_centroid_dist_to_caffeine_A < 5).sum())} of {len(rd)}")

# 3. How confident is DiffDock in its own answer?
print("\n-- DiffDock's own confidence in its top pose --")
print(f"median confidence : {rd.diffdock_top_confidence.median():.3f}")
print(f"negative (low-confidence) calls : "
      f"{int((rd.diffdock_top_confidence < 0).sum())} of {len(rd)}")

The verdict is unambiguous, and it is not the comfortable one. **Not one of the twenty ligands has its
top-ranked DiffDock pose within 2 angstroms of the DiffSBDD pose**, the threshold docking papers
conventionally treat as a successful reproduction. The median disagreement is **4.39 angstroms**. If you
are generous and let each ligand pick whichever of its ten DiffDock poses happens to be closest, the
median only improves to **3.36 angstroms** and just **2 of 20** reach 2 angstroms. Two independent methods
looking at the same molecule and the same receptor do not agree on where the molecule goes.

And yet they agree on something important. **Nineteen of twenty** re-docked ligands put their centroid
within 5 angstroms of the co-crystal caffeine, with a median of **1.77 angstroms** against **0.63
angstroms** for the generated poses. Both methods find the same pocket. The disagreement is about
orientation and detail *inside* that pocket, not about which cavity of the receptor to occupy. That is a
meaningfully better outcome than a scatter across the protein surface, and it is worth separating from the
RMSD result rather than averaging the two into a single vague impression.

DiffDock is also not confident in its own answers. Its median top-pose confidence is **0.097** and **7 of
20** calls are negative, which in DiffDock's calibration signals a pose it does not trust. So this is not
a case of a reliable method overruling an unreliable one. It is two uncertain methods disagreeing, which
is a weaker and more honest thing to report.

In [ ]:
# Same PoseBusters bar, two pose sources. Where does each one fail?
gen_path = str(DATA / "posebusters_diffsbdd.csv")
if not os.path.exists(gen_path):
    gen_path = "posebusters_diffsbdd.csv"        # fallback: local working copy
gen = read_provenanced_csv(gen_path)             # this file DOES carry '#' header lines

CHECKS = ["bond_lengths", "bond_angles", "internal_steric_clash",
          "non-aromatic_ring_non-flatness", "minimum_distance_to_protein"]

side_by_side = pd.DataFrame({
    "DiffSBDD pose (%)":   [100 * gen[c].mean() for c in CHECKS],
    "DiffDock re-dock (%)": [100 * rd[c].mean() for c in CHECKS],
}, index=CHECKS)

print(side_by_side.to_string(float_format=lambda v: f"{v:5.0f}"))
print(f"\nfully valid, DiffSBDD poses    : {int(gen.pb_valid.sum())} of {len(gen)}")
print(f"fully valid, DiffDock re-docks : {int(rd.pb_valid.sum())} of {len(rd)}")

This is the most instructive table in the section, because the two pose sources fail in **opposite**
directions under an identical bar.

**DiffSBDD's poses are well-behaved with respect to the protein and strained inside the molecule.** They
pass `minimum_distance_to_protein` on 100% of ligands, so nothing is jammed into the receptor. But they
fail `bond_angles` on 15% and `bond_lengths` on 10%: the generator is placing atoms in a pocket and
sometimes bending the molecule to make it fit.

**DiffDock's re-docked poses are the mirror image.** Bond lengths and angles pass on 100% of ligands, and
the reason is architectural: DiffDock does not invent geometry. It starts from a valid RDKit conformer and
moves it as a rigid body plus torsion rotations, so internal geometry is valid by construction. What it
gets wrong is the protein relationship; **55% of its top poses violate the minimum-distance-to-protein
check**, meaning atoms sit closer to the receptor than physics allows. On the full suite that is
**15 of 20 valid for DiffSBDD against 8 of 20 for DiffDock**.

The lesson generalises past these two tools. A single validity score hides *where* a method is weak, and
two methods with similar headline pass rates can be unreliable in completely different ways. When you swap
one component of a pipeline for another, the failure mode moves; it does not necessarily shrink.

![Figure I.18]({{artifact:1f588f05-08a2-40c3-afe6-dc6802c8c075}})

**Figure I.18.** Independent re-docking of all twenty DiffSBDD ligands with DiffDock-L, under the same
PoseBusters bar. Panel (a) contrasts per-check pass rates for the generated poses (crimson) and the
re-docked poses (slate), and the pattern is a crossover rather than one method dominating: DiffSBDD keeps
its distance from the protein but strains bond lengths and angles, while DiffDock has perfect internal
geometry and violates minimum-distance-to-protein on 55% of ligands. Panel (b) shows the distribution of
RMSD between each generated pose and its re-docked counterpart, with the conventional 2-angstrom
reproduction threshold marked; the top-ranked poses (outline) sit almost entirely to the right of it, and
taking the closest of ten poses (filled) shifts the mass left without moving many ligands across. Panel
(c) tracks each ligand's centroid distance to the 3RFM co-crystal caffeine, generated against re-docked,
where every line rises but nearly all stay under 5 angstroms. The figure separates two claims that are
easy to conflate: the two methods disagree on pose detail, and they agree on the binding site.

<div style="background:#eef4fb;border-left:5px solid #2c6fbf;padding:10px 14px;border-radius:4px;margin:6px 0;">
<b style="color:#2c6fbf;"> What this cross-check is, and what it is not</b><br>

<b>It is not a re-docking accuracy benchmark.</b> Those exist, they are valuable, and they work by
re-docking a ligand whose <i>experimental</i> crystal pose is known, then measuring how close the
prediction lands. Every one of our twenty ligands is <b>de novo</b>: it has never been synthesised and no
experimental pose exists for it. So there is no ground truth in this comparison, and <b>neither source can
be called correct</b>. When DiffSBDD and DiffDock disagree by 4.39 angstroms, that number tells you the
two methods disagree; it does not tell you which one is wrong.
<br><br>
<b>What it does buy you</b> is independent corroboration, which is the practical substitute for ground
truth when ground truth does not exist. Site agreement across two methods is real evidence about the site.
Pose disagreement is real evidence that the exact orientation is one method's opinion and should not be
carried downstream as if it were settled.
<br><br>
<b>One reproducibility note.</b> These numbers came from the <b>hosted DiffDock NIM</b>, and the model it
serves is <b>not</b> the MIT repository's checkpoint; per its model card, NVIDIA trained the served model
on PLINDER and SAIR. If you reproduce this from <code>gcorso/DiffDock</code> you are running a different
set of weights and you should expect different numbers. Same architecture lineage, different model.

</div>

## <b> <font color='#A20025'> Summary</font> </b>


You read; rather than ran; the frontier of structure-based generative design, and then you put its output
through a controlled test. Both halves matter, and the second half is where the appendix actually lands.

- **&sect;I.2; RFdiffusion.** The denoising idea you built in NB1 scales straight from a small-molecule
  atom cloud to a **protein backbone**: corrupt a real structure, learn to reverse it, sample new
  ones. RFdiffusion fine-tunes RoseTTAFold into that denoiser. The family churns fast
  (v1 to AA to v2 to v3 in about 2.5 years); RFdiffusion3's science posted as a **preprint on 2025-09-18**,
  with **open MIT code following about 2025-12**; two distinct dates.
- **&sect;I.6; Two directions.** **DiffSBDD** fixes a pocket and generates a ligand; **RFdiffusionAA**
  fixes a ligand and generates a pocket; the same denoiser, mirror-image conditioning. And
  validated design is a **pipeline, not a model**: RFdiffusion to (Ligand)ProteinMPNN to AlphaFold2 to
  PyRosetta. (See DL4Proteins WS8/9/10.)
- **&sect;I.7; Boltz-2 as connector, tested properly.** Scoring one molecule cannot tell you whether
  conditioning worked, so we scored **seven** under a byte-identical protein block on **two independent
  Boltz-2 implementations**. Pocket conditioning did help: **0.4440** for the best DiffSBDD ligand against
  **0.0906** unconditioned, a 4.9-fold gain. And **only caffeine, the real co-crystal ligand, cleared the
  0.5 binder gate**, under both implementations. The two implementations agree (Spearman **0.9643**), so
  that verdict is the model's, not the machine's.
- **&sect;I.7; A second opinion.** Re-docking all twenty DiffSBDD ligands with DiffDock-L reproduced
  **none** of the generated poses within 2 angstroms (median **4.39**), while **19 of 20** re-docks landed
  within 5 angstroms of the co-crystal caffeine. The two methods agree on the site and disagree on the pose,
  and they fail the same physical checks in opposite ways.

**So what does the whole pipeline actually deliver?** Assembled honestly, it is this. We can generate
molecules that are chemically valid, drug-like by QED, and not retrieved from the training set. We can
condition that generation on a real pocket and measurably improve how an independent affinity model ranks
the results. We can place those molecules in the pocket and show, with a rendered structure, that the
placement is chemically sensible. And then two independent checks push back: **a separate docking method
disagrees with the placement in detail while agreeing on the site, and the affinity model declines to call
any of the molecules a binder.**

That is a real result, and it is not a triumphant one. It is also the most useful ending we can give you,
because it maps exactly onto what these tools are for **today**: they are a way to *generate and rank
hypotheses*, not a way to produce compounds you would order. The generation step works. The triage step
works, which we know because it said no to us. What no part of this pipeline does is establish that a
molecule binds; that still requires an assay, and any pipeline that seems to promise otherwise is hiding a
gate that should have been shut.

**The through-line:** anchor your pipeline on small, stable, permissively-licensed models you can
actually run (FlowMol3, DiffSBDD); treat high-traffic frontier tools as **optional, version-pinned,
dated read-alongs**; and always give a generated result a second, independent opinion before you believe
it. Popularity correlates with churn and compute burden; pin it, date it, and re-verify before you run it.
Convenience correlates with borrowed trust; control it, gate it, and cross-check it before you ship it.

## <b> <font color='#A20025'> References</font> </b>


**Diffusion & flow matching (the spine you built).**
- Ho, Jain, Abbeel. *Denoising Diffusion Probabilistic Models.* NeurIPS 2020. arXiv:2006.11239.
- Lipman, Chen, Ben-Hamu, Nickel, Le. *Flow Matching for Generative Modeling.* ICLR 2023. arXiv:2210.02747.
- Dunn, Koes. *FlowMol3: Flow Matching for 3D De Novo Small-Molecule Generation.* arXiv:2508.12629, 2025. **MIT.**

**Frontier read-alongs (this notebook).**
- Watson et al. *De novo design of protein structure and function with RFdiffusion.* **Nature 620**, 2023. doi:10.1038/s41586-023-06415-8. Code: github.com/RosettaCommons/RFdiffusion (**BSD**).
- Butcher, Krishna, Mitra, ... Baker. *De novo Design of All-atom Biomolecular Interactions with RFdiffusion3.* **bioRxiv**, doi:10.1101/2025.09.18.676967. Preprint **2025-09-18** (v1) / **2025-11-19** (v2); open MIT code **~2025-12**.
- Schneuing et al. *Structure-based drug design with equivariant diffusion models* (**DiffSBDD**). **Nature Computational Science**, 2024. doi:10.1038/s43588-024-00737-x. MIT code; CrossDocked2020 data **GPL-2.0**.
- Passaro et al. *Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction.* **bioRxiv**, doi:10.1101/2025.06.14.659707. Posted **2025-06-18**. Code: github.com/jwohlwend/boltz (**boltz 2.2.1**, **MIT**). The hosted NIM endpoint used for the second implementation is governed by **NVIDIA service terms**, not by the repository's MIT licence.
- Corso, Stark, Jing, Barzilay, Jaakkola. *DiffDock: Diffusion Steps, Twists, and Turns for Molecular Docking.* ICLR 2023. arXiv:2210.01776. Code: github.com/gcorso/DiffDock (**MIT**). The re-docking cross-check used NVIDIA's hosted DiffDock-L NIM, whose **served weights are NVIDIA-trained on PLINDER and SAIR** and are therefore not the repository's checkpoints.
- Buttenschoen, Morris, Deane. *PoseBusters: AI-based docking methods fail to generate physically valid poses or generalise to novel sequences.* **Chemical Science 15**, 2024. doi:10.1039/D3SC04185A. The same dock-config bar was applied to both pose sources.

**Hands-on companion.**
- Graylab. *DL4Proteins notebooks*; WS8 (RFdiffusion), WS9 (RFdiffusion→ProteinMPNN→AF2), WS10 (binder design). github.com/Graylab/DL4Proteins-notebooks.

<div style="color:#888;font-size:12px;margin-top:10px;">
All external-tool versions, licenses, and dates retrieved 2026-07 and verified against each tool's
primary repository/preprint, with <b>code and weights licences read separately</b>; see Table I.1 in the
appendix text. Frontier-tool code blocks are <b>read-along and were not executed</b> in this appendix's
CPU environment. Every number in the controlled Boltz-2 comparison and the re-docking cross-check is a
<b>real measured result</b>, reproduced by the cached-load cells above from
<code>data/boltz2_comparison.csv</code> and <code>data/diffdock_redock_posebusters.csv</code>.
</div>